# H8. Устойчивость сигнала петли: ablation user_gen и свип начальных распределений

## Контекст

В H7 ([m2p]H7_mf_warmstart.ipynb) выяснилось, что **warm-start MLP**
выделяет петлю обратной связи статистически значимо:
- `tr(Σ̂)`: closed_loop коллапсирует на ~0.16–0.23 *сильнее* static
  (95% CI исключает 0);
- `KL(P_T‖P_0)`: closed_loop дрейфит на ~0.73–0.79 *сильнее* static
  (CI исключает 0).

При этом эффект небольшой по абсолютной величине, и cold-init MLP
по-прежнему даёт почти идентичные траектории во всех режимах с α>0.

## Гипотезы H8

**H8.1 (user-gen как синхронизатор).** Стохастическая замена 5%
пользователей каждый шаг (`GMMUserGenerator.step`) делает популяцию
во всех режимах одинаково «свежей» из исходного GMM, маскируя различия
динамики между режимами. *Прогноз:* при замораживании популяции
(`replacement_rate=0` + отключение `update_component_params_from_data`)
разрыв `closed_loop − static` по обеим метрикам расширится.

**H8.2 (зависимость от GMM).** Симметричная конфигурация (K=3,
inter_dist=5, σ=0.8) — лишь одна точка в пространстве дизайнов.
Эффект петли зависит от:
- числа кластеров `K`,
- межкластерного расстояния `inter_dist`,
- ширины кластеров `σ`,
- баланса размеров кластеров (`weights`).

*Прогноз:* в распределениях с **сильной кластерной структурой**
(K≥3, inter_dist≥5, σ≤0.8) warm-MLP видит явный разрыв; в распределениях
без структуры (K=1 — единый гауссиан, или большое σ) разрыв исчезает,
так как pretrain нечему учить.

## Дизайн

- **Часть A.** MLP-warm × {4 режима} × {full (rep=5%), frozen (rep=0)} ×
  20 seeds = 160 запусков. Контролируем замену пользователей.
- **Часть B.** MLP-warm × {4 режима} × 4 свипа × ~12 значений × 20 seeds
  ≈ 960 запусков. Свип по K, inter_dist, σ, weights — каждая ось
  меняется при дефолте остальных.

Метрики: `tr(Σ̂)_T`, `KL(P_T‖P_0)`, `gini_T`, `coverage_T`. Главный
показатель — разрыв `closed_loop − static` с бутстрап-CI.


In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from itertools import product

from sim.user_generator import GMMUserGenerator
from sim.environment    import SimulationEnvironment, ExperimentDataset
from sim.click_model    import ClickModel
from models.rec_models  import RecModel
from models.serving     import ServingPolicy

FIGURES_DIR = Path('../../paper/figures')
RESULTS_DIR = Path('results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

Imports OK


In [2]:
# ── Конфигурация ─────────────────────────────────────────────────────
SMOKE = False    # True → 1 seed, проверка пайплайна

EMB_DIM      = 8
N_USERS      = 300
N_ITEMS      = 300
K_REC        = 10
T            = 100
T_RET        = 10
REPLACE_DEF  = 0.05          # дефолтный replacement_rate (часть A: default)
ADHERENCE    = 0.7
USER_DRIFT   = 0.005
DRIFT_ALPHA  = 0.02
INTER_DIST_D = 5.0
SIGMA_K_D    = 0.8
K_GMM_D      = 3
WEIGHTS_D    = 'uniform'

# Warm-start
N_PRETRAIN_EPOCHS  = 20
PRETRAIN_LR        = 3e-3
PRETRAIN_SUBSAMPLE = 6000

N_SEEDS = 1 if SMOKE else 20

MODES  = ['closed_loop', 'static', 'fresh_oracle', 'no_influence']
COLORS = {'closed_loop': '#d62728', 'static': '#ff7f0e',
          'fresh_oracle': '#2ca02c', 'no_influence': '#1f77b4'}
LABELS = {'closed_loop': r'closed\_loop',
          'static':       'static',
          'fresh_oracle': r'fresh\_oracle',
          'no_influence': r'no\_influence ($\alpha=0$)'}

# Сетки свипов (Часть B)
K_GRID         = [1, 2, 3, 5, 8]
INTER_DIST_GRID = [2.0, 5.0, 10.0]
SIGMA_GRID     = [0.3, 0.8, 2.0]
WEIGHTS_GRID   = ['uniform', 'skewed']     # skewed = [0.6, 0.3, 0.1] для K=3

print(f'N={N_USERS}, T={T}, seeds={N_SEEDS}')
print(f'Part A: 4 modes × 2 user_gen settings × {N_SEEDS} seeds = {4*2*N_SEEDS} runs')
print(f'Part B: 4 modes × ({len(K_GRID)}+{len(INTER_DIST_GRID)}+{len(SIGMA_GRID)}'
      f'+{len(WEIGHTS_GRID)}) values × {N_SEEDS} seeds = '
      f'{4*(len(K_GRID)+len(INTER_DIST_GRID)+len(SIGMA_GRID)+len(WEIGHTS_GRID))*N_SEEDS} runs')

N=300, T=100, seeds=20
Part A: 4 modes × 2 user_gen settings × 20 seeds = 160 runs
Part B: 4 modes × (5+3+3+2) values × 20 seeds = 1040 runs


## Дефолтная конфигурация

| Параметр | Значение | Смысл |
|----------|---------|-------|
| `N_USERS, N_ITEMS` | 300, 300 | как в H1/H7 |
| `T, T_RET` | 100, 10 | как в H1/H7 |
| `ADHERENCE` (α) | 0.7 | сила петли в `ClickModel` |
| `USER_DRIFT` (β) | 0.005 | β-дрейф эмбеддингов |
| `DRIFT_ALPHA` | 0.02 | дрейф GMM-центров (closed_loop) |
| `REPLACE_DEF` | 0.05 | дефолт; в Части A варьируется → 0 (frozen) |
| `K_GMM_D, INTER_DIST_D, SIGMA_K_D` | 3, 5, 0.8 | дефолты GMM; в Части B варьируются |
| `N_PRETRAIN_EPOCHS` | 20 | warm-start на `true_pref` (MLP) |
| `N_SEEDS` | 20 | бутстрап по запускам |

В Часть B меняется ровно один параметр GMM за раз; остальные = дефолт.
Это удерживает анализ интерпретируемым (без полного декартова произведения).


In [3]:
# ── Утилиты ─────────────────────────────────────────────────────────
def make_gmm_params(K, dim, inter_dist, sigma, weights_kind='uniform'):
    means, covs = [], []
    for k in range(K):
        angle = 2 * np.pi * k / max(K, 1)
        m = np.zeros(dim)
        m[0] = inter_dist * np.cos(angle)
        m[1] = inter_dist * np.sin(angle)
        means.append(m)
        covs.append(sigma**2 * np.eye(dim))
    if weights_kind == 'uniform':
        weights = [1.0 / K] * K
    elif weights_kind == 'skewed':
        # Гео-убывающая последовательность, нормированная
        raw = np.array([0.5**k for k in range(K)])
        weights = (raw / raw.sum()).tolist()
    else:
        raise ValueError(weights_kind)
    return means, covs, weights


def make_items(N_items, K, means, sigma, dim, rng, weights=None):
    weights = weights if weights is not None else [1.0/K]*K
    counts  = np.round(np.array(weights) * N_items).astype(int)
    counts[-1] = N_items - counts[:-1].sum()
    parts = []
    for k in range(K):
        n_k = max(counts[k], 1)
        parts.append(rng.multivariate_normal(means[k], sigma**2 * np.eye(dim), n_k))
    return np.vstack(parts)


def make_true_pref(users, items, tau=2.0):
    scores = users @ items.T / tau
    return 1 / (1 + np.exp(-scores))


class _PairBCEDataset(Dataset):
    def __init__(self, users_emb, items_emb, target_matrix, subsample, rng):
        n_u, n_i = target_matrix.shape
        all_pairs = np.array([(u, i) for u in range(n_u) for i in range(n_i)])
        if subsample is not None and subsample < len(all_pairs):
            idx = rng.choice(len(all_pairs), subsample, replace=False)
            all_pairs = all_pairs[idx]
        self.pairs    = all_pairs.astype(np.int64)
        self.targets  = target_matrix[self.pairs[:, 0], self.pairs[:, 1]].astype(np.float32)
        self.users_emb = users_emb.astype(np.float32)
        self.items_emb = items_emb.astype(np.float32)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return (torch.from_numpy(self.users_emb[u]),
                torch.from_numpy(self.items_emb[i]),
                torch.tensor(self.targets[idx]))


def pretrain_model(model, user_emb, item_emb, target_matrix,
                   n_epochs, lr, subsample, rng, device):
    ds = _PairBCEDataset(user_emb, item_emb, target_matrix, subsample, rng)
    loader = DataLoader(ds, batch_size=512, shuffle=True)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCELoss()
    model.train()
    for _ in range(n_epochs):
        for u_b, i_b, t_b in loader:
            u_b, i_b, t_b = u_b.to(device), i_b.to(device), t_b.to(device)
            opt.zero_grad()
            pred = model(u_b, i_b)
            loss = crit(pred, t_b)
            loss.backward(); opt.step()


def freeze_user_gen(user_gen):
    """Монки-патч: user_gen.step становится no-op, drift параметров отключается."""
    def noop_step(dataset, t):
        return (user_gen.embeddings.copy(),
                user_gen.cluster_labels.copy(),
                dataset.matrix,
                np.array([], dtype=int))
    def noop_update():
        pass
    user_gen.step = noop_step
    user_gen.update_component_params_from_data = noop_update


def build_env(mode, seed, *, K_GMM, INTER_DIST, SIGMA_K, WEIGHTS_KIND,
              REPLACE_RATE, frozen_users=False):
    np.random.seed(seed)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed + 1000)

    means, covs, weights = make_gmm_params(K_GMM, EMB_DIM, INTER_DIST, SIGMA_K,
                                            weights_kind=WEIGHTS_KIND)
    gen = GMMUserGenerator(
        component_means=means, component_covs=covs, component_weights=weights,
        replacement_rate=REPLACE_RATE, memory_effect=6)

    np.random.seed(seed)
    user_emb, _ = gen.initialize(N_USERS)
    item_emb = make_items(N_ITEMS, K_GMM, means, SIGMA_K, EMB_DIM, rng, weights)
    true_pref = make_true_pref(user_emb, item_emb)
    matrix    = np.full((N_USERS, N_ITEMS), np.nan)
    dataset   = ExperimentDataset(user_emb.copy(), item_emb.copy(), matrix)

    device = torch.device('cpu')
    model  = RecModel(EMB_DIM, EMB_DIM, hidden_size=64).to(device)
    # warm-start всегда (фокус H8 — MLP-warm)
    pretrain_model(model, user_emb, item_emb, true_pref,
                   n_epochs=N_PRETRAIN_EPOCHS, lr=PRETRAIN_LR,
                   subsample=PRETRAIN_SUBSAMPLE, rng=rng, device=device)

    if frozen_users:
        freeze_user_gen(gen)

    policy = ServingPolicy('top_k')
    alpha_c = 0.0 if mode == 'no_influence' else ADHERENCE
    click   = ClickModel(adherence=alpha_c, usage_rate=0.8, noise_level=0.05)
    d_alpha = DRIFT_ALPHA if mode == 'closed_loop' else 0.0

    env = SimulationEnvironment(
        dataset=dataset, rec_model=model,
        user_generator=gen, click_model=click,
        serving_policy=policy, mode=mode,
        true_preference_matrix=true_pref,
        retrain_period=T_RET, K=K_REC,
        device=device, seen_filter=True,
        user_drift_beta=USER_DRIFT,
        drift_alpha=d_alpha)
    return env


def run_one(mode, seed, *, frozen_users, K_GMM=None, INTER_DIST=None,
            SIGMA_K=None, WEIGHTS_KIND=None, REPLACE_RATE=None):
    env = build_env(
        mode, seed,
        K_GMM=K_GMM if K_GMM is not None else K_GMM_D,
        INTER_DIST=INTER_DIST if INTER_DIST is not None else INTER_DIST_D,
        SIGMA_K=SIGMA_K if SIGMA_K is not None else SIGMA_K_D,
        WEIGHTS_KIND=WEIGHTS_KIND if WEIGHTS_KIND is not None else WEIGHTS_D,
        REPLACE_RATE=REPLACE_RATE if REPLACE_RATE is not None else REPLACE_DEF,
        frozen_users=frozen_users)
    for t in range(T):
        env.step(t)
    return env.metrics.get_dataframe()

print('Utility functions ready')

Utility functions ready


## Часть A. Замена пользователей как синхронизатор режимов

Сравниваем `replacement_rate = 0.05` (дефолт) с *замороженной* популяцией
(монки-патч `user_gen.step → no-op`, без `update_component_params_from_data`).
В дефолте 5% × 100 шагов = ~500 замен (популяция 300, т.е. ~1.6 полной
смены за прогон). Заморозка устраняет общий внешний источник
стохастичности.


In [4]:
# ── Часть A. user_gen ablation ──────────────────────────────────────
resA = {}   # key = (frozen, mode) -> list[DataFrame]
combos = list(product([False, True], MODES))
total_A = len(combos) * N_SEEDS
t_start = time.time(); run_idx = 0
for frozen, mode in combos:
    key = (frozen, mode)
    resA[key] = []
    for seed in range(N_SEEDS):
        run_idx += 1; t0 = time.time()
        df = run_one(mode, seed, frozen_users=frozen)
        df['frozen'] = frozen; df['mode'] = mode; df['seed'] = seed
        resA[key].append(df)
        if seed == 0 or seed == N_SEEDS-1:
            tot = time.time() - t_start
            eta = tot / run_idx * (total_A - run_idx)
            tag = 'FROZEN' if frozen else 'full  '
            print(f'  A[{run_idx:3d}/{total_A}] {tag} {mode:14s} seed={seed:2d} '
                  f'trΣ→{df.trace_sigma.iloc[-1]:5.2f}  KL→{df.kl_from_initial.iloc[-1]:5.2f} '
                  f'({time.time()-t0:.1f}s, ETA {eta/60:.1f}m)')
print(f'\nPart A done in {(time.time()-t_start)/60:.1f} min')

  A[  1/160] full   closed_loop    seed= 0 trΣ→ 1.41  KL→16.35 (10.4s, ETA 27.6m)
  A[ 20/160] full   closed_loop    seed=19 trΣ→ 1.13  KL→16.21 (10.0s, ETA 24.0m)
  A[ 21/160] full   static         seed= 0 trΣ→ 1.37  KL→15.77 (8.8s, ETA 23.7m)
  A[ 40/160] full   static         seed=19 trΣ→ 1.30  KL→15.82 (8.9s, ETA 19.2m)
  A[ 41/160] full   fresh_oracle   seed= 0 trΣ→ 1.15  KL→16.34 (9.3s, ETA 19.0m)
  A[ 60/160] full   fresh_oracle   seed=19 trΣ→ 1.08  KL→16.14 (9.6s, ETA 15.9m)
  A[ 61/160] full   no_influence   seed= 0 trΣ→ 7.95  KL→ 7.49 (9.3s, ETA 15.7m)
  A[ 80/160] full   no_influence   seed=19 trΣ→ 8.41  KL→ 7.86 (9.2s, ETA 12.6m)
  A[ 81/160] FROZEN closed_loop    seed= 0 trΣ→ 0.51  KL→ 9.63 (6.6s, ETA 12.4m)
  A[100/160] FROZEN closed_loop    seed=19 trΣ→ 0.52  KL→ 9.64 (6.5s, ETA 8.8m)
  A[101/160] FROZEN static         seed= 0 trΣ→ 0.50  KL→ 9.66 (5.9s, ETA 8.7m)
  A[120/160] FROZEN static         seed=19 trΣ→ 0.52  KL→ 9.67 (5.7s, ETA 5.5m)
  A[121/160] FROZEN fresh_ora

In [5]:
# ── Часть A. Анализ ─────────────────────────────────────────────────
def bootstrap_gap(arr_cl, arr_st, n_boot=2000, q=(0.025, 0.975), rng=None):
    rng = rng or np.random.default_rng(0)
    cl, st = np.asarray(arr_cl), np.asarray(arr_st)
    if len(cl) == 1 and len(st) == 1:
        d = cl[0] - st[0]
        return d, d, d
    idx = rng.integers(0, len(cl), size=(n_boot, len(cl)))
    diffs = cl[idx].mean(1) - st[idx].mean(1)
    return float(diffs.mean()), float(np.quantile(diffs, q[0])), float(np.quantile(diffs, q[1]))

rng_b = np.random.default_rng(0)
A_rows = []
for frozen in [False, True]:
    for metric in ['trace_sigma', 'kl_from_initial']:
        cl = [df[metric].iloc[-1] for df in resA[(frozen, 'closed_loop')]]
        st = [df[metric].iloc[-1] for df in resA[(frozen, 'static')]]
        fo = [df[metric].iloc[-1] for df in resA[(frozen, 'fresh_oracle')]]
        ni = [df[metric].iloc[-1] for df in resA[(frozen, 'no_influence')]]
        gap_cs, lo_cs, hi_cs = bootstrap_gap(cl, st, rng=rng_b)
        gap_cn, lo_cn, hi_cn = bootstrap_gap(cl, ni, rng=rng_b)
        A_rows.append(dict(
            frozen=frozen, metric=metric,
            mean_cl=np.mean(cl), mean_st=np.mean(st),
            mean_fo=np.mean(fo), mean_ni=np.mean(ni),
            gap_cl_minus_st=gap_cs, lo_cs=lo_cs, hi_cs=hi_cs,
            gap_cl_minus_ni=gap_cn, lo_cn=lo_cn, hi_cn=hi_cn,
            sig_cs=lo_cs * hi_cs > 0,
            sig_cn=lo_cn * hi_cn > 0))
A_df = pd.DataFrame(A_rows)
A_df.to_csv(RESULTS_DIR / 'H8_partA_user_gen_ablation.csv', index=False)
print('Part A summary:')
print(A_df.round(3).to_string(index=False))

Part A summary:
 frozen          metric  mean_cl  mean_st  mean_fo  mean_ni  gap_cl_minus_st  lo_cs  hi_cs  gap_cl_minus_ni   lo_cn   hi_cn  sig_cs  sig_cn
  False     trace_sigma    1.215    1.392    1.214    8.534           -0.177 -0.221 -0.129           -7.322  -7.828  -6.867    True    True
  False kl_from_initial   16.405   15.616   16.113    7.582            0.790  0.670  0.913            8.826   8.468   9.181    True    True
   True     trace_sigma    0.516    0.512    0.506   17.232            0.004  0.003  0.006          -16.717 -16.867 -16.558    True    True
   True kl_from_initial    9.623    9.627    9.684    1.976           -0.005 -0.016  0.011            7.647   7.623   7.670   False    True


In [6]:
# ── Часть A. Рисунок: gap closed_loop−static, frozen vs full ────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, metric, title in zip(
        axes, ['trace_sigma', 'kl_from_initial'],
        [r'$\mathrm{tr}(\hat\Sigma_T)$:  closed\_loop $-$ static',
         r'$KL(P_T\|P_0)$:  closed\_loop $-$ static']):
    rows = A_df[A_df.metric == metric].sort_values('frozen')
    x = np.arange(len(rows))
    vals = rows['gap_cl_minus_st'].values
    los  = np.maximum(vals - rows['lo_cs'].values, 0)
    his  = np.maximum(rows['hi_cs'].values - vals, 0)
    colors = ['#4c72b0' if not f else '#dd8452' for f in rows.frozen]
    ax.bar(x, vals, yerr=[los, his], color=colors, capsize=4, alpha=0.85)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(['full (rep=5%)', 'frozen (rep=0)'], fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.grid(True, axis='y', ls='--', alpha=0.4)
fig.suptitle('H8.A: эффект петли при замороженной vs стандартной замене пользователей')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H8_partA_user_gen_ablation.pdf', bbox_inches='tight')
plt.show()
print('Saved H8_partA_user_gen_ablation.pdf')

Saved H8_partA_user_gen_ablation.pdf


## Часть B. Свип по начальным распределениям

Для каждой оси меняем один параметр, остальные = дефолт. На каждой
конфигурации запускаем 4 режима × 20 seeds и считаем `closed_loop − static`
гэп с бутстрап-CI.


In [7]:
# ── Часть B. Свип ───────────────────────────────────────────────────
sweep_specs = [
    ('K',          K_GRID,           'K_GMM'),
    ('inter_dist', INTER_DIST_GRID,  'INTER_DIST'),
    ('sigma',      SIGMA_GRID,       'SIGMA_K'),
    ('weights',    WEIGHTS_GRID,     'WEIGHTS_KIND'),
]
total_B = sum(len(grid) for _, grid, _ in sweep_specs) * len(MODES) * N_SEEDS
print(f'Total Part B runs: {total_B}')

resB = {}   # key = (axis_name, value, mode) -> list[DataFrame]
t_start = time.time(); run_idx = 0
for axis_name, grid, kw in sweep_specs:
    for val in grid:
        for mode in MODES:
            key = (axis_name, val, mode)
            resB[key] = []
            for seed in range(N_SEEDS):
                run_idx += 1; t0 = time.time()
                kwargs = {kw: val, 'frozen_users': False}
                df = run_one(mode, seed, **kwargs)
                df['axis'] = axis_name; df['val'] = str(val); df['mode'] = mode; df['seed'] = seed
                resB[key].append(df)
                if seed == 0 or seed == N_SEEDS-1:
                    tot = time.time() - t_start
                    eta = tot / run_idx * (total_B - run_idx)
                    print(f'  B[{run_idx:3d}/{total_B}] {axis_name}={str(val):<8s} '
                          f'{mode:14s} seed={seed:2d} '
                          f'trΣ→{df.trace_sigma.iloc[-1]:5.2f}  '
                          f'KL→{df.kl_from_initial.iloc[-1]:5.2f}  '
                          f'({time.time()-t0:.1f}s, ETA {eta/60:.1f}m)')
print(f'\nPart B done in {(time.time()-t_start)/60:.1f} min')

Total Part B runs: 1040
  B[  1/1040] K=1        closed_loop    seed= 0 trΣ→ 0.03  KL→20.82  (7.0s, ETA 121.5m)
  B[ 20/1040] K=1        closed_loop    seed=19 trΣ→ 0.03  KL→20.06  (6.9s, ETA 118.8m)
  B[ 21/1040] K=1        static         seed= 0 trΣ→ 0.07  KL→20.26  (5.7s, ETA 117.7m)
  B[ 40/1040] K=1        static         seed=19 trΣ→ 0.08  KL→20.56  (5.7s, ETA 106.2m)
  B[ 41/1040] K=1        fresh_oracle   seed= 0 trΣ→ 0.01  KL→21.99  (10.7s, ETA 107.8m)
  B[ 60/1040] K=1        fresh_oracle   seed=19 trΣ→ 0.01  KL→22.80  (10.6s, ETA 127.4m)
  B[ 61/1040] K=1        no_influence   seed= 0 trΣ→ 0.07  KL→20.57  (5.7s, ETA 126.8m)
  B[ 80/1040] K=1        no_influence   seed=19 trΣ→ 0.07  KL→20.59  (5.7s, ETA 116.5m)
  B[ 81/1040] K=2        closed_loop    seed= 0 trΣ→ 1.85  KL→17.18  (9.6s, ETA 116.8m)
  B[100/1040] K=2        closed_loop    seed=19 trΣ→ 1.53  KL→17.75  (9.8s, ETA 122.1m)
  B[101/1040] K=2        static         seed= 0 trΣ→ 1.80  KL→17.23  (8.5s, ETA 122.1m)
  B[12

In [8]:
# ── Часть B. Агрегация ──────────────────────────────────────────────
rng_b = np.random.default_rng(7)
B_rows = []
for axis_name, grid, _ in sweep_specs:
    for val in grid:
        cl = [df['trace_sigma'].iloc[-1] for df in resB[(axis_name, val, 'closed_loop')]]
        st = [df['trace_sigma'].iloc[-1] for df in resB[(axis_name, val, 'static')]]
        fo = [df['trace_sigma'].iloc[-1] for df in resB[(axis_name, val, 'fresh_oracle')]]
        ni = [df['trace_sigma'].iloc[-1] for df in resB[(axis_name, val, 'no_influence')]]
        kl_cl = [df['kl_from_initial'].iloc[-1] for df in resB[(axis_name, val, 'closed_loop')]]
        kl_st = [df['kl_from_initial'].iloc[-1] for df in resB[(axis_name, val, 'static')]]
        kl_ni = [df['kl_from_initial'].iloc[-1] for df in resB[(axis_name, val, 'no_influence')]]
        tr_gap, tr_lo, tr_hi = bootstrap_gap(cl, st, rng=rng_b)
        kl_gap, kl_lo, kl_hi = bootstrap_gap(kl_cl, kl_st, rng=rng_b)
        B_rows.append(dict(
            axis=axis_name, value=str(val),
            trace_cl=np.mean(cl), trace_st=np.mean(st),
            trace_fo=np.mean(fo), trace_ni=np.mean(ni),
            kl_cl=np.mean(kl_cl), kl_st=np.mean(kl_st), kl_ni=np.mean(kl_ni),
            trace_gap=tr_gap, trace_lo=tr_lo, trace_hi=tr_hi,
            kl_gap=kl_gap, kl_lo=kl_lo, kl_hi=kl_hi,
            sig_trace=tr_lo*tr_hi > 0, sig_kl=kl_lo*kl_hi > 0))
B_df = pd.DataFrame(B_rows)
B_df.to_csv(RESULTS_DIR / 'H8_partB_distribution_sweep.csv', index=False)
print(B_df.round(3).to_string(index=False))

      axis   value  trace_cl  trace_st  trace_fo  trace_ni  kl_cl  kl_st  kl_ni  trace_gap  trace_lo  trace_hi  kl_gap  kl_lo  kl_hi  sig_trace  sig_kl
         K       1     0.029     0.074     0.009     0.073 20.424 20.640 20.605     -0.045    -0.049    -0.042  -0.212 -0.404 -0.017       True    True
         K       2     1.715     1.734     1.364     6.611 17.593 17.038 15.186     -0.020    -0.111     0.065   0.556  0.339  0.775      False    True
         K       3     1.215     1.392     1.214     8.534 16.405 15.616  7.582     -0.177    -0.225    -0.132   0.791  0.658  0.917       True    True
         K       5     1.244     1.375     0.860     8.032 17.323 16.891 12.265     -0.132    -0.203    -0.065   0.433  0.310  0.551       True    True
         K       8     1.125     1.307     0.745     9.443 17.156 16.807  8.846     -0.182    -0.244    -0.125   0.350  0.237  0.462       True    True
inter_dist     2.0     0.181     0.282     0.153     0.761 18.289 17.033 10.582     -0.1

In [9]:
# ── Часть B. Рисунки: 4 оси × 2 метрики ────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(17, 8), sharey='row')
for c, (axis_name, grid, _) in enumerate(sweep_specs):
    sub = B_df[B_df.axis == axis_name].reset_index(drop=True)
    x = np.arange(len(sub))
    for r, (metric_col, gap_col, lo_col, hi_col, ylabel) in enumerate([
        ('trace_sigma', 'trace_gap', 'trace_lo', 'trace_hi',
         r'gap $\mathrm{tr}\hat\Sigma_T$:  closed\_loop $-$ static'),
        ('kl_from_initial', 'kl_gap', 'kl_lo', 'kl_hi',
         r'gap $KL_T$:  closed\_loop $-$ static'),
    ]):
        ax = axes[r, c]
        vals = sub[gap_col].values
        los = np.maximum(vals - sub[lo_col].values, 0)
        his = np.maximum(sub[hi_col].values - vals, 0)
        # цвет: значимый — синий, не значимый — серый
        sig_col = 'sig_trace' if metric_col == 'trace_sigma' else 'sig_kl'
        colors = ['#4c72b0' if s else '#999999' for s in sub[sig_col]]
        ax.bar(x, vals, yerr=[los, his], color=colors, capsize=3, alpha=0.85)
        ax.axhline(0, color='k', lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels(sub['value'].values, fontsize=9)
        ax.set_xlabel(axis_name)
        if c == 0:
            ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(True, axis='y', ls='--', alpha=0.4)
        if r == 0:
            ax.set_title(f'свип по {axis_name}', fontsize=10)
fig.suptitle('H8.B: разрыв (closed\_loop $-$ static) по 4 осям начального распределения '
             '(MLP-warm, синий — 95% CI исключает 0)', y=1.00)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H8_partB_distribution_sweep.pdf', bbox_inches='tight')
plt.show()
print('Saved H8_partB_distribution_sweep.pdf')

Saved H8_partB_distribution_sweep.pdf


In [10]:
# ── Дополнительный рисунок: trajectories по K (наглядно) ────────────
# Покажем tr(Σ) во времени для K ∈ {1, 3, 8} × 4 режима
selected_K = [1, 3, 8]
fig, axes = plt.subplots(1, len(selected_K), figsize=(14, 4), sharey=True)
for ax, K in zip(axes, selected_K):
    for mode in MODES:
        key = ('K', K, mode)
        if key not in resB:
            continue
        arr = np.stack([df['trace_sigma'].values for df in resB[key]])
        ts  = resB[key][0]['t'].values
        mu, sd = arr.mean(0), arr.std(0)
        ax.plot(ts, mu, color=COLORS[mode], lw=1.8, label=LABELS[mode])
        ax.fill_between(ts, mu - sd, mu + sd, color=COLORS[mode], alpha=0.15)
    ax.set_title(f'K = {K}', fontsize=11)
    ax.set_xlabel('Step $t$')
    ax.grid(True, ls='--', alpha=0.4)
    if K == selected_K[0]:
        ax.set_ylabel(r'$\mathrm{tr}(\hat{\Sigma}_t^u)$')
        ax.legend(fontsize=8, loc='upper right')
fig.suptitle('H8.B: динамика tr(Σ̂) при разном числе GMM-кластеров (MLP-warm)')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H8_K_trajectories.pdf', bbox_inches='tight')
plt.show()
print('Saved H8_K_trajectories.pdf')

Saved H8_K_trajectories.pdf


## Интерпретация

### Часть A (`H8_partA_user_gen_ablation.pdf`)

- Если **frozen — full разрыв расширяется** (т.е. `gap_cl_minus_st`
  стало больше по модулю), это подтверждает H8.1: замена пользователей
  смешивает поведение режимов. В этом случае рекомендуется
  переформулировать «контрольный эксперимент H1» с замороженной
  популяцией.
- Если разрыв **не изменился** — user_gen не является синхронизатором;
  идентичные траектории были побочным эффектом β-дрейфа (как в §19
  главного отчёта).

### Часть B (`H8_partB_distribution_sweep.pdf` + `H8_K_trajectories.pdf`)

- **K (число кластеров).** Если разрыв растёт с K — петля сильнее
  проявляется в популяциях со сложной кластерной структурой. Случай
  K=1 (без кластеров) — самый чистый тест: если разрыв здесь = 0, это
  значит, что warm-pretrain в принципе не учит ничего, что отличало
  бы пользователей.
- **inter_dist.** Большое расстояние = чёткая структура. Прогнозируется
  монотонное расширение разрыва с inter_dist.
- **σ (ширина кластеров).** Большое σ ≈ K=1 (все кластеры слиты);
  малое σ ≈ дискретные точки. Ожидается убывание разрыва с σ.
- **weights.** При сильном дисбалансе (`skewed`) меньшие кластеры
  получают меньше показов и могут не успевать дрейфить — это даст
  более «глобальный» эффект петли (на доминирующем кластере).

### Связь с текстом ВКР

Результаты H8 идут в **главу 3** как раздел «Чувствительность
сигнала петли к параметрам моделирования». Аргументация для текста:

1. В H7 был обнаружен значимый эффект петли при warm-start MLP.
2. В H8 показано, в каких условиях этот эффект сохраняется и где
   разрушается.
3. Это даёт **границы применимости** утверждений H1/H3: эффект петли
   реален, но требует специфических условий (структура распределения
   + информированный рекомендатель), что согласуется с переформулировкой
   H3 в §7.3 [REPORT.md](diagnostic/REPORT.md).
